In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from collections import deque
import random

In [ ]:
# -----------------------------# Extended compound encoding for multiple compounds
compound_mapping = {
    "HARD": 0,
    "MEDIUM": 1,
    "SOFT": 2,
    # Add more mappings as needed
}

# State representation: select relevant features from the dataset.
# Here we include: LapNumber, Sector1Time, Sector2Time, Sector3Time, TyreLife,
# Compound, Position, TimeGapToLeader, TimeGapToBehind, AirTemp, TrackTemp,
# SpeedI1, SpeedI2.
class RaceState:
    def __init__(self, row):
        self.lap_number = row["LapNumber"] if pd.notna(row["LapNumber"]) else 0.0
        self.sector1 = row["Sector1Time"] if pd.notna(row["Sector1Time"]) else 0.0
        self.sector2 = row["Sector2Time"] if pd.notna(row["Sector2Time"]) else 0.0
        self.sector3 = row["Sector3Time"] if pd.notna(row["Sector3Time"]) else 0.0
        self.tyre_life = row["TyreLife"] if pd.notna(row["TyreLife"]) else 0.0
        compound_str = row["Compound"].strip().upper() if pd.notna(row["Compound"]) else "UNKNOWN"
        self.compound = compound_mapping.get(compound_str, -1)
        self.position = row["Position"] if pd.notna(row["Position"]) else 0.0
        self.time_gap_leader = row["TimeGapToLeader"] if pd.notna(row["TimeGapToLeader"]) else 0.0
        self.time_gap_behind = row["TimeGapToBehind"] if pd.notna(row["TimeGapToBehind"]) else 0.0
        self.air_temp = row["AirTemp"] if pd.notna(row["AirTemp"]) else 0.0
        self.track_temp = row["TrackTemp"] if pd.notna(row["TrackTemp"]) else 0.0
        self.speed_i1 = row["SpeedI1"] if pd.notna(row["SpeedI1"]) else 0.0
        self.speed_i2 = row["SpeedI2"] if pd.notna(row["SpeedI2"]) else 0.0

    def to_array(self):
        return np.array([
            self.lap_number, self.sector1, self.sector2, self.sector3,
            self.tyre_life, self.compound, self.position,
            self.time_gap_leader, self.time_gap_behind,
            self.air_temp, self.track_temp,
            self.speed_i1, self.speed_i2
        ], dtype=np.float32)

# Define a simple binary action space: NO_PIT and PIT_STOP.
class RaceAction:
    NO_PIT = 0
    PIT_STOP = 1

    @staticmethod
    def get_action_space():
        return [RaceAction.NO_PIT, RaceAction.PIT_STOP]

# -----------------------------
# Environment Definitions
# -----------------------------

# Base environment for the race dataset.
# Each row of the CSV represents one lap.
class RaceEnvironment:
    def __init__(self, filename):
        self.data = pd.read_csv(filename)
        self.total_laps = len(self.data)
        self.current_index = 0
        self.state = None

    def reset(self):
        self.current_index = 0
        self.state = RaceState(self.data.iloc[self.current_index])
        return self.state

    def step(self, action):
        row = self.data.iloc[self.current_index]
        lap_time = row["LapTime"]
        pit_time = row["PitTime"] if pd.notna(row["PitTime"]) else 0.0
        # If a pit stop is taken, add pit penalty to lap time.
        effective_lap_time = lap_time + pit_time if action == RaceAction.PIT_STOP else lap_time
        # Modified reward: higher reward for lower lap times.
        reward = 120 - effective_lap_time
        self.current_index += 1
        done = (self.current_index >= self.total_laps)
        if not done:
            self.state = RaceState(self.data.iloc[self.current_index])
        else:
            self.state = None
        return self.state, reward, done

# DRQN benefits from sequence information.
# This wrapper collects a fixed-length sequence of states.
class RaceEnvironmentSeq:
    def __init__(self, env, seq_length=5):
        self.env = env
        self.seq_length = seq_length
        self.state_seq = deque(maxlen=seq_length)

    def reset(self):
        state = self.env.reset()
        # Fill the sequence with the initial state repeated.
        self.state_seq = deque([state for _ in range(self.seq_length)], maxlen=self.seq_length)
        return self.get_state_seq()

    def step(self, action):
        next_state, reward, done = self.env.step(action)
        self.state_seq.append(next_state)
        return self.get_state_seq(), reward, done

    def get_state_seq(self):
        # Convert the sequence of RaceState objects to a numpy array.
        # If a state is None (episode finished), fill with zeros.
        seq = [
            s.to_array() if s is not None 
            else np.zeros_like(self.state_seq[0].to_array())
            for s in self.state_seq
        ]
        return np.array(seq, dtype=np.float32)

# -----------------------------
# DRQN Model and Training
# -----------------------------

# Build the DRQN model using an LSTM to process the sequence.
def build_drqn_model(seq_length, feature_size, action_space_size):
    model = Sequential()
    model.add(LSTM(64, input_shape=(seq_length, feature_size), return_sequences=False))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(action_space_size, activation='linear'))
    model.compile(optimizer='adam', loss='mse')
    return model

# Replay Buffer stores experience tuples: (state_seq, action, reward, next_state_seq, done).
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

# Training loop for DRQN with sequences, replay buffer, and a target network.
def train_drqn(env_seq, episodes, gamma, epsilon, epsilon_decay, min_epsilon,
               batch_size=32, replay_capacity=1000, target_update_freq=5):
    # Get state shape from an initial sequence.
    initial_seq = env_seq.reset()
    seq_length, feature_size = initial_seq.shape
    action_space = RaceAction.get_action_space()
    action_space_size = len(action_space)

    # Build the main and target networks.
    model = build_drqn_model(seq_length, feature_size, action_space_size)
    target_model = build_drqn_model(seq_length, feature_size, action_space_size)
    target_model.set_weights(model.get_weights())

    replay_buffer = ReplayBuffer(replay_capacity)

    for episode in range(episodes):
        state_seq = env_seq.reset()  # Shape: (seq_length, feature_size)
        total_reward = 0
        done = False

        while not done:
            state_input = state_seq.reshape(1, seq_length, feature_size)
            # Epsilon-greedy action selection.
            if np.random.rand() < epsilon:
                action = np.random.choice(action_space)
            else:
                q_values = model.predict(state_input, verbose=0)
                action = np.argmax(q_values[0])
            next_state_seq, reward, done = env_seq.step(action)
            total_reward += reward

            # Store experience.
            replay_buffer.add((state_seq, action, reward, next_state_seq, done))
            state_seq = next_state_seq

            # Batch update when enough samples are available.
            if len(replay_buffer) >= batch_size:
                batch = replay_buffer.sample(batch_size)
                state_batch = np.array([exp[0] for exp in batch])
                target_batch = model.predict(state_batch, verbose=0)

                for i, (s_seq, a, r, s_next_seq, d) in enumerate(batch):
                    if d:
                        target_batch[i][a] = r
                    else:
                        next_input = s_next_seq.reshape(1, seq_length, feature_size)
                        t = target_model.predict(next_input, verbose=0)
                        target_batch[i][a] = r + gamma * np.amax(t[0])
                model.fit(state_batch, target_batch, epochs=1, verbose=0)

        epsilon = max(min_epsilon, epsilon * epsilon_decay)
        print(f"Episode {episode+1}/{episodes}, Total Reward: {total_reward}")

        # Update target network weights periodically.
        if (episode+1) % target_update_freq == 0:
            target_model.set_weights(model.get_weights())

In [ ]:
filename = r"..\Datasets\2024\ver_Abu Dhabi Grand Prix_2024_laps.csv"  # Ensure this CSV is in your working directory.
base_env = RaceEnvironment(filename)
seq_length = 5  # Define the length of the state sequence for the LSTM.
env_seq = RaceEnvironmentSeq(base_env, seq_length=seq_length)

# Hyperparameters for training.
episodes = 2
gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.995
min_epsilon = 0.01

train_drqn(env_seq, episodes, gamma, epsilon, epsilon_decay, min_epsilon)

In [ ]:
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, LSTM
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
import pickle
import matplotlib.pyplot as plt

# Define the binary action space.
class RaceAction:
    NO_PIT = 0
    PIT_STOP = 1

    @staticmethod
    def get_action_space():
        return [RaceAction.NO_PIT, RaceAction.PIT_STOP]

# Build the DRQN model.
def build_drqn_model(input_shape, action_space_size):
    model = Sequential()
    model.add(LSTM(64, input_shape=input_shape, return_sequences=False))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(action_space_size, activation='linear'))
    model.compile(optimizer='adam', loss='mse')
    return model

# Load and preprocess the dataset from CSV.
def load_dataset(filename):
    df = pd.read_csv(filename)
    
    # Encode compound: HARD -> 0, others -> 1.
    def encode_compound(compound):
        return 0 if compound.strip().upper() == "HARD" else 1

    states = []
    actions = []
    rewards = []
    lap_numbers = []
    for i in range(len(df) - 1):
        row = df.iloc[i]
        state = np.array([
            row["LapNumber"],
            row["Sector1Time"],
            row["Sector2Time"],
            row["Sector3Time"],
            row["TyreLife"],
            encode_compound(row["Compound"]),
            row["Position"],
            row["TimeGapToLeader"],
            row["TimeGapToBehind"],
            row["AirTemp"],
            row["TrackTemp"]
        ], dtype=np.float32)
        states.append(state)
        lap_numbers.append(row["LapNumber"])
        # Convert PitStop column to action: "true" means pit stop (1), else 0.
        act = 1 if str(row["PitStop"]).strip().lower() == "true" else 0
        actions.append(act)
        pit_time = row["PitTime"] if pd.notna(row["PitTime"]) else 0.0
        effective_lap_time = row["LapTime"] + pit_time
        rewards.append(120 - effective_lap_time)
        
    states = np.array(states)
    actions = np.array(actions)
    rewards = np.array(rewards)
    
    # Prepare next_states and terminal flags.
    next_states = states[1:]
    dones = np.zeros(len(states))
    dones[-1] = 1  # Mark the last sample as terminal.
    
    # Align states with next_states.
    states = states[:-1]
    actions = actions[:-1]
    rewards = rewards[:-1]
    dones = dones[:-1]
    lap_numbers = lap_numbers[:-1]

    # Scale states and next_states.
    scaler = StandardScaler()
    states = scaler.fit_transform(states)
    next_states = scaler.transform(next_states)

    return states, actions, rewards, next_states, dones, scaler, lap_numbers, df

# Train the DRQN model using Q-learning update rules.
def train_drqn_model(filename, episodes=10, gamma=0.99, batch_size=32):
    states, actions, rewards, next_states, dones, scaler, lap_numbers, df = load_dataset(filename)
    input_shape = (1, states.shape[1])  # LSTM expects 3D input: (timesteps, features)
    action_space_size = len(RaceAction.get_action_space())
    model = build_drqn_model(input_shape, action_space_size)
    
    # Split the dataset into training and validation sets.
    X_train, X_val, a_train, a_val, r_train, r_val, ns_train, ns_val, d_train, d_val = train_test_split(
        states, actions, rewards, next_states, dones, test_size=0.2, random_state=42
    )
    
    # Reshape for LSTM input.
    X_train = X_train.reshape(-1, 1, states.shape[1])
    X_val = X_val.reshape(-1, 1, states.shape[1])
    ns_train = ns_train.reshape(-1, 1, states.shape[1])
    ns_val = ns_val.reshape(-1, 1, states.shape[1])
    
    for episode in range(episodes):
        print(f"\nEpisode {episode + 1}/{episodes}")
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_train = X_train[indices]
        a_train = a_train[indices]
        r_train = r_train[indices]
        ns_train = ns_train[indices]
        d_train = d_train[indices]
        
        for i in range(0, X_train.shape[0], batch_size):
            batch_states = X_train[i:i + batch_size]
            batch_actions = a_train[i:i + batch_size]
            batch_rewards = r_train[i:i + batch_size]
            batch_next_states = ns_train[i:i + batch_size]
            batch_dones = d_train[i:i + batch_size]
            
            current_q_values = model.predict(batch_states, verbose=0)
            next_q_values = model.predict(batch_next_states, verbose=0)
            
            targets = current_q_values.copy()
            for j in range(len(batch_states)):
                act = batch_actions[j]
                if batch_dones[j]:
                    targets[j][act] = batch_rewards[j]
                else:
                    targets[j][act] = batch_rewards[j] + gamma * np.max(next_q_values[j])
            model.fit(batch_states, targets, epochs=1, verbose=0)
        
        val_loss = model.evaluate(X_val, np.zeros((len(X_val), action_space_size)), verbose=0)
        print(f"Validation Loss: {val_loss}")

    val_q_values = model.predict(X_val, verbose=0)
    predicted_actions_val = np.argmax(val_q_values, axis=1)
    accuracy = accuracy_score(a_val, predicted_actions_val)
    conf_matrix = confusion_matrix(a_val, predicted_actions_val)

    print("\nFinal Evaluation on Validation Set:")
    print(f"Accuracy: {accuracy}")
    print("Confusion Matrix:")
    print(conf_matrix)
    
    # Save the trained model and scaler.
    model.save("drqn_race_strategy_model.h5")
    print("Model saved to drqn_race_strategy_model.h5")
    with open("scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)
    print("Scaler saved to scaler.pkl")
    
    return model, scaler, lap_numbers, df

# Visualize results using matplotlib in separate windows and print pit stop statements.
def visualize_results(model, scaler, df):
    # Recreate state vectors from CSV data.
    states = []
    for i in range(len(df) - 1):
        row = df.iloc[i]
        compound_encoded = 0 if row["Compound"].strip().upper() == "HARD" else 1
        state = np.array([
            row["LapNumber"],
            row["Sector1Time"],
            row["Sector2Time"],
            row["Sector3Time"],
            row["TyreLife"],
            compound_encoded,
            row["Position"],
            row["TimeGapToLeader"],
            row["TimeGapToBehind"],
            row["AirTemp"],
            row["TrackTemp"]
        ], dtype=np.float32)
        states.append(state)
    states = np.array(states)
    states_scaled = scaler.transform(states)
    states_reshaped = states_scaled.reshape(-1, 1, states_scaled.shape[1])
    
    # Predict Q-values and derive pit stop actions.
    q_values = model.predict(states_reshaped, verbose=0)
    predicted_actions = np.argmax(q_values, axis=1)
    
    # Create a DataFrame for visualization (ensuring alignment with predictions).
    df_vis = df.iloc[:len(predicted_actions)].copy()
    df_vis["PredictedAction"] = predicted_actions
    # Convert the actual PitStop column to a numeric flag.
    df_vis["ActualPitStop"] = df_vis["PitStop"].apply(lambda x: 1 if str(x).strip().lower() == "true" else 0)
    
    # --- Print Actual Pit Stop Statements ---
    actual_pit = df_vis[df_vis["ActualPitStop"] == 1]
    print("\nActual Pit Stops:")
    for idx, row in actual_pit.iterrows():
        print(f"Lap {row['LapNumber']}: Lap Time = {row['LapTime']} sec, Compound = {row['Compound']}")
    
    # --- Print Predicted Pit Stop Statements ---
    predicted_pit = df_vis[df_vis["PredictedAction"] == RaceAction.PIT_STOP]
    print("\nPredicted Pit Stops:")
    for idx, row in predicted_pit.iterrows():
        print(f"Lap {row['LapNumber']}: Lap Time = {row['LapTime']} sec, Compound = {row['Compound']}")
    
    # --- Figure 1: Actual Race Lap Times with Actual Pit Stops ---
    plt.figure(figsize=(10, 6))
    plt.plot(df_vis["LapNumber"], df_vis["LapTime"], label="Lap Time", color="blue", linestyle="-", marker="o")
    plt.scatter(actual_pit["LapNumber"], actual_pit["LapTime"], color="red", s=100, label="Actual Pit Stop")
    plt.title("Actual Race Lap Times with Actual Pit Stops")
    plt.xlabel("Lap Number")
    plt.ylabel("Lap Time (seconds)")
    plt.legend()
    plt.grid(True)
    
    # --- Figure 2: Race Lap Times with Predicted Pit Stops ---
    plt.figure(figsize=(10, 6))
    plt.plot(df_vis["LapNumber"], df_vis["LapTime"], label="Lap Time", color="blue", linestyle="-", marker="o")
    plt.scatter(predicted_pit["LapNumber"], predicted_pit["LapTime"], color="green", s=100, label="Predicted Pit Stop")
    plt.title("Race Lap Times with Predicted Pit Stops")
    plt.xlabel("Lap Number")
    plt.ylabel("Lap Time (seconds)")
    plt.legend()
    plt.grid(True)
    
    # Display the graphs in separate windows.
    plt.show()



In [ ]:
filename = r"..\Datasets\2024\ver_Abu Dhabi Grand Prix_2024_laps.csv" # Replace with your CSV file path.
episodes = 10
gamma = 0.99
batch_size = 32

model, scaler, lap_numbers, df = train_drqn_model(filename, episodes, gamma, batch_size)
visualize_results(model, scaler, df)